In [1]:
import json
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedShuffleSplit
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import precision_score, recall_score, roc_auc_score, accuracy_score, f1_score

In [2]:
def cross_validation_rf(model_id, model_type):
    with open(f'results/{model_id}_{model_type}_best_within_one.json', 'r') as f:
        params = json.load(f)
    f.close()
    del params["achieved_value"]
    params["n_estimators"] = 250
    int_param_list = ["max_depth", "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf", "max_leaf_nodes"]
    for key, value in params.items():
        if key in int_param_list:
            params[key] = int(value)
        else:
            params[key] = value
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    # define metric
    metric_list = ['precision', "recall", "roc_auc", "accuracy", "f1"]
    # set model
    model = RandomForestClassifier(**params)
    X = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()
    cv_test = cross_validate(model, X, y, cv=cv, scoring=metric_list)
    # print result
    print(model_id, model_type)
    print("Recall: ", cv_test["test_recall"].mean().round(3))
    print("Roc: ", cv_test["test_roc_auc"].mean().round(3))
    print("Accuracy: ", cv_test["test_accuracy"].mean().round(3))
    # save results
    with open(f"results/{model_id}_{model_type}.pkl", "wb") as f:
        pickle.dump(cv_test, f)

In [3]:
cross_validation_rf("model_1","rf")

model_1 rf
Recall:  0.844
Roc:  0.909
Accuracy:  0.835


In [4]:
cross_validation_rf("model_2","rf")

model_2 rf
Recall:  0.867
Roc:  0.822
Accuracy:  0.758


In [5]:
def cross_validation_rf_us(model_id, model_type):
    with open(f'results/{model_id}_{model_type}_best_within_one.json', 'r') as f:
        params = json.load(f)
    f.close()
    del params["achieved_value"]
    params["n_estimators"] = 250
    int_param_list = ["max_depth", "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf", "max_leaf_nodes"]
    for key, value in params.items():
        if key in int_param_list:
            params[key] = int(value)
        else:
            params[key] = value
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    # define metric
    metric_list = ['precision', "recall", "roc_auc", "accuracy", "f1"]
    # set model
    model = RandomForestClassifier(**params)
    X = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()
    cv_test = {
        "fold": [],
        'test_precision': [],
        'test_recall': [],
        'test_roc_auc': [],
        'test_accuracy': [],
        'test_f1': []
    }
    fold_indices = list(cv.split(X, y))
    data_balancer = RandomOverSampler()
    for current_fold in range(len(fold_indices)):
        cv_test['fold'].append(current_fold)
        train_idx, val_idx = fold_indices[current_fold]
        X_train, y_train = X.iloc[train_idx], y[train_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]
        X_train_balanced, y_train_balanced = data_balancer.fit_resample(X_train, y_train)
        fitted_model = model.fit(X_train_balanced, y_train_balanced)
        y_pred = fitted_model.predict(X_val)
        y_pred_proba = fitted_model.predict_proba(X_val)
        cv_test['test_precision'].append(precision_score(y_val, y_pred))
        cv_test['test_recall'].append(recall_score(y_val, y_pred))
        cv_test['test_roc_auc'].append(roc_auc_score(y_val, y_pred_proba[:, 1]))
        cv_test['test_accuracy'].append(accuracy_score(y_val, y_pred))
        cv_test['test_f1'].append(f1_score(y_val, y_pred))
    for metric in ['test_precision', 'test_recall', 'test_roc_auc', 'test_accuracy', 'test_f1']:
        cv_test[metric] = np.array(cv_test[metric])

    # print result
    print(model_id, model_type)
    print("Recall: ", cv_test["test_recall"].mean().round(3))
    print("Roc: ", cv_test["test_roc_auc"].mean().round(3))
    print("Accuracy: ", cv_test["test_accuracy"].mean().round(3))
    # save results
    with open(f"results/{model_id}_{model_type}.pkl", "wb") as f:
        pickle.dump(cv_test, f)

In [6]:
cross_validation_rf_us("model_1a","rf")

model_1a rf
Recall:  0.741
Roc:  0.851
Accuracy:  0.774


In [7]:
cross_validation_rf_us("model_2a","rf")

model_2a rf
Recall:  0.741
Roc:  0.783
Accuracy:  0.712


In [8]:
cross_validation_rf_us("model_1b","rf")

model_1b rf
Recall:  0.873
Roc:  0.888
Accuracy:  0.838


In [9]:
cross_validation_rf_us("model_2b","rf")

model_2b rf
Recall:  0.763
Roc:  0.829
Accuracy:  0.754
